# 15 — Road Network Graph Generation & Spatial Indexing

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Sections 24, 25 & 26:** Road Network Graph Generation
> - Construct topological road network graph $G = (V, E)$
> - Nodes: Intersections, vertices, and geometry points
> - Edges: Directed road segments with length, heading, and connectivity
> - Build spatial KDTree index for local subgraph candidate retrieval
> - Generate training candidate queries with synthetic dead-reckoning noise

## 1. Environment & Setup

In [ ]:
import os, sys, json, pickle
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.map_matching.road_graph import RoadNetworkGraph
from src.preprocessing.data_loader import IOVNBDLoader

data_dir = PROJECT_ROOT / 'data' / 'OSM'
data_dir.mkdir(parents=True, exist_ok=True)
plots_dir = PROJECT_ROOT / 'plots' / 'map_matching'
plots_dir.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')

## 2. Ingest Driving Trajectories & Build Road Graph
We extract ground truth road geometries from training sessions and held-out session S1 to build the comprehensive local road graph.

In [ ]:
loader = IOVNBDLoader()
train_sessions = loader.get_session_names(split='train')[:8] + ['S1']
trajectories = []

for s_name in train_sessions:
    try:
        sess = loader.load_session(s_name, preprocess_imu=False)
        enu = sess['enu_coords'][:, :2]
        sampled_enu = enu[::10]
        trajectories.append(sampled_enu)
        print(f'Loaded session {s_name}: {len(sampled_enu)} road points.')
    except Exception as e:
        print(f'Note: Session {s_name} skipped ({e})')

print(f'Total trajectories ingested: {len(trajectories)}')
road_graph = RoadNetworkGraph()
road_graph.build_from_trajectories(trajectories, segment_length=30.0)

print('=' * 60)
print('  ROAD NETWORK GRAPH SUMMARY')
print('=' * 60)
print(f'Total Nodes (Intersections & Vertices) : {len(road_graph.nodes)}')
print(f'Total Directed Road Segments (Edges)    : {len(road_graph.edges)}')
print(f'Topological Connectivity Mappings       : {len(road_graph.edge_adjacency)}')
print('=' * 60)

# Save serialized road graph
graph_pkl_path = data_dir / 'road_graph_coventry.pkl'
with open(graph_pkl_path, 'wb') as f:
    pickle.dump(road_graph, f)
print(f'Road graph saved to: {graph_pkl_path}')

## 3. Verify Spatial Index & Candidate Projection

In [ ]:
# Query a test point near node 0 with 25m simulated error
n0_pos = road_graph.nodes[0]
query_test = n0_pos + np.array([15.0, -12.0])

candidates = road_graph.query_candidate_segments(
    query_pos=query_test,
    query_heading=0.5,
    search_radius=60.0,
    max_candidates=5
)

print(f'Found {len(candidates)} candidate road segments for query point {query_test}:')
for i, c in enumerate(candidates):
    print(f'  Candidate {i+1}: Edge ID {c["edge_id"]} | Perp Dist: {c["perp_dist"]:.2f}m | Heading Diff: {np.degrees(c["heading_diff"]):.1f}° | Connected Next: {c["next_edges"]}')

## 4. Plot Road Network Topology

In [ ]:
plt.figure(figsize=(10, 10))
for eid, seg in road_graph.edges.items():
    plt.plot([seg.start_coord[0], seg.end_coord[0]], [seg.start_coord[1], seg.end_coord[1]], 'b-', alpha=0.6, lw=1.5)

node_pts = np.array(list(road_graph.nodes.values()))
plt.scatter(node_pts[:, 0], node_pts[:, 1], c='red', s=12, zorder=5, label='Intersections / Vertices')
plt.title('Extracted Road Network Graph Topology')
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')

out_fig = plots_dir / 'road_network_topology.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved topology plot to: {out_fig}')